In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
%matplotlib inline

## 1. Carga y normalizacion del stream

In [ ]:
# Carga el dump de ticks del WebSocket
df = pd.read_csv('data/stream.csv')
print('Columnas crudas (formato corto Finnhub):', df.columns.tolist())
df.head()

In [ ]:
# Renombrado Finnhub usa llaves cortas p/s/t/v
df = df.rename(columns={'p': 'price', 's': 'symbol', 't': 'timestamp_ms', 'v': 'volume'})
df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms', utc=True)
df['symbol'] = df['symbol'].str.replace('OANDA:', '', regex=False).str.replace('_', '', regex=False)
df = df[['timestamp', 'symbol', 'price', 'volume']].dropna(subset=['timestamp', 'symbol', 'price'])
print(f'Total de ticks: {len(df)}')
print(f'Símbolos únicos: {df.symbol.nunique()}')
print(f'Rango temporal: {df.timestamp.min()} → {df.timestamp.max()}')
df.head()

## 2. Calidad y caracteristicas del stream

In [ ]:
# Distribucion de mensajes por simbolo
conteo_simbolo = df['symbol'].value_counts()
print('Ticks recibidos por símbolo:')
print(conteo_simbolo)

fig, ax = plt.subplots(figsize=(10, 4))
conteo_simbolo.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Volumen de ticks recibidos por par de divisa')
ax.set_xlabel('Símbolo')
ax.set_ylabel('Cantidad de ticks')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

df_sorted = df.sort_values('timestamp').reset_index(drop=True)
df_sorted['delta_ms'] = df_sorted['timestamp'].diff().dt.total_seconds() * 1000

print(f'Mediana entre ticks: {df_sorted.delta_ms.median():.1f} ms')
print(f'Promedio entre ticks: {df_sorted.delta_ms.mean():.1f} ms')
print(f'Percentil 95: {df_sorted.delta_ms.quantile(0.95):.1f} ms')
print(f'Máximo (gap): {df_sorted.delta_ms.max():.1f} ms = {df_sorted.delta_ms.max()/1000:.1f} s')

In [ ]:
# Histograma 
fig, ax = plt.subplots(figsize=(12, 4))
df_sorted['delta_ms'].clip(upper=2000).hist(bins=60, ax=ax, color='teal', edgecolor='white')
ax.set_title('Distribución del tiempo entre ticks (≤ 2 s)')
ax.set_xlabel('Milisegundos entre ticks')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

## 3. Comportamiento por simbolo

In [ ]:

top_symbol = conteo_simbolo.idxmax()
print(f'Símbolo seleccionado: {top_symbol}')

df_top = df[df['symbol'] == top_symbol].sort_values('timestamp').copy()
df_top['minute'] = df_top['timestamp'].dt.floor('min')
df_top.head()

In [ ]:
# ticks po minuto
ticks_por_minuto = df_top.groupby('minute').size()
print(f'Promedio de ticks por minuto: {ticks_por_minuto.mean():.1f}')
print(f'Máximo en un minuto: {ticks_por_minuto.max()}')
print(f'Mínimo en un minuto: {ticks_por_minuto.min()}')

fig, ax = plt.subplots(figsize=(14, 4))
ticks_por_minuto.plot(ax=ax, color='teal', linewidth=1.5)
ax.set_title(f'Ticks por minuto - {top_symbol}')
ax.set_xlabel('Minuto')
ax.set_ylabel('Cantidad de ticks')
plt.tight_layout()
plt.show()

In [ ]:
# Aplicamos transformación OHLC 
ohlc = (
    df_top.groupby('minute')['price']
    .agg(open='first', high='max', low='min', close='last')
    .reset_index()
)
ohlc['spread'] = ohlc['high'] - ohlc['low']
print(f'Velas OHLC generadas: {len(ohlc)}')
ohlc.head()

In [ ]:
#  ticks crudos vs velas OHLC agregadas
fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(df_top['timestamp'], df_top['price'], s=8, alpha=0.4, color='gray', label='Ticks crudos')
ax.plot(ohlc['minute'], ohlc['close'], color='navy', linewidth=1.5, label='Close por minuto (agregado)')
ax.fill_between(ohlc['minute'], ohlc['low'], ohlc['high'], alpha=0.25, color='steelblue', label='Rango high-low')
ax.set_title(f'{top_symbol} - Ticks crudos vs agregación OHLC por minuto')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Precio')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# dispersión del spread
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(ohlc['spread'], bins=40, color='teal', edgecolor='white')
axes[0].set_title(f'Distribución del spread intra-minuto - {top_symbol}')
axes[0].set_xlabel('high - low')

axes[1].boxplot(ohlc['spread'], vert=False)
axes[1].set_title('Boxplot del spread')
axes[1].set_xlabel('high - low')
plt.tight_layout()
plt.show()

print(f'Spread promedio: {ohlc.spread.mean():.6f}')
print(f'Spread P95: {ohlc.spread.quantile(0.95):.6f}')
print(f'Spread máx: {ohlc.spread.max():.6f}')

## 4. Comparación de volatilidad entre múltiples símbolos

In [ ]:
# Calculamos OHLC para todos los símbolos 
validos = conteo_simbolo[conteo_simbolo > 50].index.tolist()
print(f'Símbolos con >50 ticks: {validos}')

resultados = []
for sym in validos:
    sub = df[df['symbol'] == sym].copy()
    sub['minute'] = sub['timestamp'].dt.floor('min')
    o = sub.groupby('minute')['price'].agg(['min', 'max', 'mean'])
    o['spread'] = o['max'] - o['min']
    resultados.append({
        'symbol': sym,
        'ticks': len(sub),
        'precio_medio': sub['price'].mean(),
        'spread_medio': o['spread'].mean(),
        'spread_pct': o['spread'].mean() / sub['price'].mean() * 100,
    })
resumen = pd.DataFrame(resultados).sort_values('spread_pct', ascending=False)
resumen

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(resumen['symbol'], resumen['spread_pct'], color='teal')
ax.set_title('Volatilidad intra-minuto comparada (spread % promedio)')
ax.set_xlabel('Símbolo')
ax.set_ylabel('Spread medio / precio medio (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()